In [3]:
from apps.importss import *

ModuleNotFoundError: No module named 'apps'

In [4]:
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool


In [5]:
from pydantic import BaseModel , Field 
import httpx

In [6]:
import os 

CAL_API_KEY = os.getenv("CAL_API_KEY")
HEADERS  = {
    "Authorization" :f"Bearer {CAL_API_KEY}",
    "call-api-version" : "2024-09-13" , 
    "content-type" : "application/json"
}

res = httpx.get("https://api.cal.com/v2/event-types", headers=HEADERS)
print(res.json())

{'status': 'success', 'data': {'eventTypeGroups': [{'teamId': None, 'bookerUrl': 'https://cal.com', 'membershipRole': None, 'profile': {'slug': 'mayank-agnmlx', 'name': 'Mayank', 'image': 'https://lh3.googleusercontent.com/a/ACg8ocJ9-InFlofn1U8QTmAYNU2cntDEweZGrjDSZPMXcPEThOKnH_Yf=s96-c'}, 'eventTypes': [{'id': 5182102, 'teamId': None, 'schedulingType': None, 'userId': 2300921, 'metadata': None, 'description': None, 'interfaceLanguage': None, 'hidden': True, 'slug': '30min', 'length': 30, 'title': '30 min meeting', 'requiresConfirmation': False, 'canSendCalVideoTranscriptionEmails': True, 'requiresConfirmationForFreeEmail': False, 'requiresConfirmationWillBlockSlot': False, 'autoTranslateDescriptionEnabled': False, 'position': 0, 'offsetStart': 0, 'owner': {'timeZone': 'Asia/Calcutta'}, 'profileId': None, 'eventName': None, 'parentId': None, 'timeZone': None, 'periodType': 'UNLIMITED', 'periodStartDate': None, 'periodEndDate': None, 'periodDays': None, 'periodCountCalendarDays': None, 

In [7]:
# Define the agent tools
from datetime import datetime, timedelta
CAL_API_KEY = os.getenv("CAL_API_KEY")
EVENT_TYPE_ID = 5182102       
YOUR_NAME = "Mayank"
YOUR_EMAIL = "mkdogra1981@gmail.com"
YOUR_TIMEZONE = "Asia/Kolkata"

HEADERS  = {
    "Authorization" :f"Bearer {CAL_API_KEY}",
    "call-api-version" : "2024-09-13" , 
    "content-type" : "application/json"
}

BASE = "https://api.cal.com/v2"


In [8]:
@tool
def create_calender_event(title:str,start_time:str,duration_minutes:int=30,description : str="")-> str:
    """ 
     Create an event on the calendar (syncs to Google Calendar automatically).
    start_time must be ISO 8601 format e.g. '2025-04-01T10:00:00Z'
    """
    if len(start_time)==10:
        start_time=start_time+"T10:00:00+05:30"




    
    payload = {
    "eventTypeId": int(EVENT_TYPE_ID),  # force int

    "start": start_time,  # must be full ISO

    "timeZone": "Asia/Kolkata",
    "language": "en",

    "metadata": {},

    "responses": {
        "name": YOUR_NAME,
        "email": YOUR_EMAIL,
        "location": {
            "value": "Google Meet"
        }
    }
}

    

    res = httpx.post(f"{BASE}/bookings",json=payload,   headers=HEADERS)
    data = res.json()
    if data.get("status") == "success":
        uid = data["data"]["uid"]
        return f"Event Created: Booking UID:{uid}.It Will Appear In Your Google Calendar"
    return f"failed to create event :{data}"


In [9]:
@tool
def list_upcoming_event()->str:
    """list all upcoming calendar events/bookings"""

    res = httpx.get(f"{BASE}/bookings?status=upcoming",headers=HEADERS)
    data = res.join()
    if data.get("status") != "success":
        return f"Error :{data}"
    bookings = data["data"]
    if not bookings:
        return "No upcoming events found."
    
    results=[]
    for b in bookings:
        results.append(f"- [{b['uid']}] {b.get('title','Event')} at {b['start']}")
        return "\n".join(results)

In [ ]:
def cancel_calendar_event(booking_uid:str,reason:str="cancelled by agent")->str:
    """ Cancel/delete a calendar event by its booking uid,
    use list_upcoming_event first to get the uid"""
    payload = {"cancellation":reason}
    res = httpx.post(f"{BASE}/bookings/{booking_uid}/cancel",json=payload,headers=HEADERS)
    data = res.json()
    if data.get("status") =="success":
        return f"EVENT {booking_uid} cancelled and removed from the google calendar."
    return f"failed to cancel :{data}"


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
     api_key=os.getenv("OPENROUTER_API_KEY"),
     model="qwen/qwen-2.5-7b-instruct",
     streaming=True
 )
from langchain.agents import create_agent
agent = create_agent(
    model=llm,
    tools=[create_calender_event, list_upcoming_event, cancel_calendar_event],
    system_prompt="""You are a personal calendar assistant for Mayank.

You have exactly 3 tools:

1. create_calendar_event — create a new event
2. list_upcoming_events — show upcoming events
3. cancel_calendar_event — cancel an event by UID

Your job is to act quickly and intelligently, not to ask unnecessary questions.

---

DECISION LOGIC:

1. If the user wants to create/add/schedule something → call create_calendar_event
2. If the user wants to view/check schedule → call list_upcoming_events
3. If the user wants to cancel/delete/remove → call cancel_calendar_event
4. Otherwise → respond normally (no tool)

---

SMART EVENT CREATION RULES:

* Extract:
  • title
  • date
  • time (if available)

* If TIME is missing:
  → default to 10:00 AM (Asia/Kolkata)

* If DATE is relative (e.g., tomorrow, next Monday):
  → convert it into exact YYYY-MM-DD

* Always generate full ISO format:
  → YYYY-MM-DDTHH:MM:SS+05:30

* Timezone is ALWAYS:
  → Asia/Kolkata

* Language is ALWAYS:
  → en

---

CRITICAL BEHAVIOR RULES:

* DO NOT ask for confirmation if enough info exists

* DO NOT ask follow-up questions if defaults can be applied

* ONLY ask questions if absolutely necessary (missing BOTH date and time)

* NEVER mention:
  • booking UID
  • raw JSON
  • API errors

---

RESPONSE STYLE:

* Keep responses short and natural
* Sound like a smart assistant, not a form
* Example:
  “Your event ‘bday’ is scheduled for April 2nd at 10 AM.”

---

ERROR HANDLING:

* If tool fails:
  → say: “Something went wrong while creating the event. Please try again.”
  → DO NOT expose technical details

---

IMPORTANT:

You are proactive. You complete tasks with best assumptions instead of asking repeatedly.

""")

In [ ]:
query = input("ask anything:")
while query:
    res = agent.invoke({"messages":[{"role":"user","content":query}]})
    ress = res["messages"][-1].content
    print(ress)

Something went wrong while creating the event. Please try again.
Something went wrong while creating the event. Please try again.
Something went wrong while creating the event. Please try again.


KeyboardInterrupt: 

In [ ]:
print("EMAIL:", YOUR_EMAIL)


EMAIL: None


In [10]:
from pathlib import Path 
from datetime import datetime , timedelta
import os

from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.oauth2.credentials import credentials 

SCOPES = ['https://www.googleapis.com/auth/calendar']

# BASE_DIR = Path(__file__).resolve().parent.parent (its for .py and not .ipynb)
BASE_DIR  = Path.cwd()
CREDS_PATH = BASE_DIR/ "credentials.json"
TOKEN_PATH = BASE_DIR/"token.json"


def get_services():
    creds = None

    # Load SAVED LOGIN
    if TOKEN_PATH.exists():
        creds = credentials.from_authorized_user_file(TOKEN_PATH,SCOPES)
    # if not logged in login
    if not creds or creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file(str(CREDS_PATH),SCOPES)
        creds = flow.run_local_server(port=0)

In [11]:
def create_event(title:str,date:str):
    service = get_services()

    start = datetime.fromisoformat(date+"T10:00:00")
    end = start + timedelta(hours=1)

    events = {
        "summary":title,
        "start":{
            "datetime":start.isoformat(),
            "timezone" : "Asia/kolkata"
        },
        "end" :{
            "datetime" :end.isoformat(),
            "timezone" : "Asia/kolkata"
        },
    }

    event = service.events().insert(
        calendarid = "primary",
        body = event
    ).execute

    return f"Event : '{title}' creatd on {date}"


In [ ]:
def list_events():
    service = get_services()

    now = datetime.utcnow().isoformat() + "Z"

    events_result = service.events().list(
        CalendarId = "primary",
        timeMin = now,
        maxResults = 10,
        singleEvents =True,
        orderBy = "starttime"

    ).execute()

    events = events_result.get("items",[])

    if not events:
        return "no upcoming events found."
    
    output = []
    for e in events:
        start = e['start'].get("datetime",e["start"].get('date'))
        output.append(f"{e['summary']} at {start} ")

    return "\n".join(output)


In [18]:
def delete_event(event_name  : str):
    service = get_services()
    events = service.events().list(calendarId="primary").execute().get("items",[])
    for e in events:
        if e.get("summary","" ).lower()==event_name.lower():
            service.events().delete(
                calendarId="primary",
                eventId = ["id"]
            ).execute()
            return f"Deleted Event '{event_name}"
    return "EVENT NOT FOUND."


In [19]:
from langchain.tools import tool

@tool
def create_calendar_event(title: str, date: str):
    """Create event. Date must be YYYY-MM-DD"""
    return create_event(title, date)


@tool
def list_calendar_events():
    """List upcoming events"""
    return list_events()


@tool
def delete_calendar_event(title: str):
    """Delete event by title"""
    return delete_event(title)

In [22]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
llm = ChatGroq(model= "llama-3.3-70b-versatile")
agent = create_agent(
    model=llm,
    tools=[create_calendar_event,list_calendar_events,delete_calendar_event],
    system_prompt=""" You are a smart personal calendar assistant for Mayank.

You have exactly 3 tools:

1. create_calendar_event — create a new event
2. list_calendar_events — show upcoming events
3. delete_calendar_event — delete an event by title

Your goal is to complete tasks efficiently without unnecessary questions.

---

DECISION LOGIC:

1. If the user wants to add/create/schedule something → call create_calendar_event
2. If the user wants to view/check/list schedule → call list_calendar_events
3. If the user wants to delete/remove/cancel → call delete_calendar_event
4. Otherwise → respond normally without tools

---

EVENT CREATION RULES:

* Extract:
  • title (event name)
  • date

* If time is NOT provided:
  → default to 10:00 AM

* Convert all dates into:
  → YYYY-MM-DD format

* Examples:
  • “2nd April” → 2026-04-02
  • “tomorrow” → calculate actual date
  • “next monday” → calculate correct date

* Always pass:
  → title and date only (tool handles time)

---

CRITICAL BEHAVIOR:

* DO NOT ask for confirmation if enough info exists

* DO NOT ask follow-up questions if date + title are clear

* ONLY ask if date is completely missing

* NEVER expose:
  • raw API responses
  • internal IDs
  • JSON

---

RESPONSE STYLE:

* Keep responses short and natural
* Speak like a real assistant

Examples:

* “Your event ‘bday’ has been added on April 2 at 10 AM.”
* “You have 2 upcoming events.”
* “I’ve deleted your ‘bday’ event.”

---

ERROR HANDLING:

* If something fails:
  → say: “Something went wrong. Please try again.”
  → do NOT show technical details

---

IMPORTANT:

You are proactive. You complete tasks using reasonable defaults instead of asking repeatedly.
"""
)

In [ ]:
query = input("Ask Anything: ")
while query:
    res = agent.invoke({"messages":[{"role":"user","content":query}]})
    ress = res["messages"][-1].content
    print(ress)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=689060886301-89tiv7g3et1askqo8u2jt4tlo1gorpjb.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A53732%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=pFbW3uYYRClKnQEEJpDH4NRq0lPBZm&code_challenge=3j-dJYcQjwnNTeq_lcOj-83Trv3FWvX-UdYf-K8Qvtg&code_challenge_method=S256&access_type=offline
